# DeepDetect - entraînement du modèle sur Colab

Ce notebook entraîne un classifieur **réel / fake** à partir d'un réseau pré-entraîné (ConvNeXt-tiny), sur le dataset *140k Real and Fake Faces*, avec des **dégradations** (JPEG, flou, réduction) pour imiter des photos de messagerie.

**Avant de commencer**
1. Menu **Exécution > Modifier le type d'exécution > GPU T4**, puis enregistrer.
2. Lance les cellules **une par une**, dans l'ordre (`Maj + Entrée`).
3. Ne saute pas l'essai rapide (cellule 5) : il vérifie toute la chaîne en quelques minutes.

**Limites à garder en tête**
- Le dataset ne contient que des faux visages **StyleGAN** : le test final de ce notebook ne dit rien sur les générateurs récents ni sur tes photos. Le vrai juge est `evaluate.py` sur ton jeu de test.
- Les images réelles viennent de FFHQ (licences à usage **non commercial**) : c'est acceptable pour apprendre et prototyper, **à revérifier avant toute publication commerciale**.

In [ ]:
# 1. Vérifier que le GPU est bien actif
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4")

In [ ]:
# 2. Brancher Google Drive : le modèle y sera sauvegardé, même si Colab se déconnecte
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Récupérer le code du projet et installer les dépendances
!rm -rf deepfake-detection
!git clone --depth 1 https://github.com/RipsySais/deepfake-detection.git
%cd deepfake-detection
!pip install -q -r requirements-train.txt

**4. Essai rapide (smoke test)** : quelques dizaines d'images, une seule epoch. Il télécharge le dataset (plusieurs Go, quelques minutes la première fois), entraîne, sauvegarde, puis recharge le modèle comme le fera l'application. À la fin, tu dois voir une ligne `Contrôle du modèle sauvegardé`.

Si une erreur apparaît, **copie le message complet** et envoie-le moi : on ne lance pas le long entraînement avant que cet essai passe.

In [ ]:
# 5. Essai rapide
!python train.py --smoke --output /content/smoke_model

**6. Vrai entraînement.** 10 000 images par classe (réelles et fausses), 3 epochs. Regarde, pendant l'essai précédent, le temps d'une epoch pour estimer la durée ; si c'est trop long pour la limite gratuite de Colab, baisse `--max-per-class` (par exemple 5000).

Le meilleur modèle est sauvegardé sur ton Drive après chaque epoch améliorée.

In [ ]:
# 7. Entraînement
!python train.py --max-per-class 10000 --epochs 3 --output /content/drive/MyDrive/deepdetect/deepfake_v1

In [ ]:
# 8. Préparer le téléchargement du modèle (un seul fichier zip)
!cd /content/drive/MyDrive/deepdetect && zip -r -q deepfake_v1.zip deepfake_v1
from google.colab import files
files.download('/content/drive/MyDrive/deepdetect/deepfake_v1.zip')

## Et ensuite, sur ton PC

1. Décompresse le zip dans le dossier du projet : `models/deepfake_v1/`
2. Lance l'évaluation sur **tes** images (venv activé) :

```bash
python evaluate.py --real data/eval/real --fake data/eval/fake --model models/deepfake_v1 --csv res_v1.csv
```

3. Compare avec le modèle pré-entraîné d'origine (spécificité 35,7 %, AUC 0,47) et envoie le rapport.